# 🤖 Data Analysis Agent v2
### Upload ANY Kaggle CSV → Auto-clean → Auto-analyse → Auto-plot

**Author:** Sahil Prakash — B.Tech CSE, Galgotias University

> 100% free. No API key. Fully adapts to whatever CSV you upload.

---

### Fixed in v2
- ✅ No fake sample data overwriting your upload
- ✅ Fully adapts to your actual column names and types
- ✅ Smarter plot selection based on what columns exist
- ✅ Works with any Kaggle dataset

### Run order
1. Cell 1 — install & import
2. Cell 2 — upload your CSV (only this cell, nothing overwrites it)
3. Cell 3 — agent inspects + plans
4. Cell 4 — agent cleans
5. Cell 5 — agent analyses + plots
6. Cell 6 — export cleaned CSV + report
7. Cell 7 — custom queries


## Cell 1 — Install & import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings, os, io
from datetime import datetime
from IPython.display import display, HTML
from google.colab import files

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#FAFAF8',
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : '500',
})

PALETTE = ['#534AB7','#0F6E56','#D85A30','#185FA5','#BA7517','#A32D2D','#3B6D11','#993556']
print("✅ Ready.")


## Cell 2 — Upload your Kaggle CSV

Run this cell → file picker opens → select your CSV → done.
Nothing else will overwrite this.


In [ ]:
print("📂 Click 'Choose Files' and select your Kaggle CSV...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please run this cell again and select a file.")

filename  = list(uploaded.keys())[0]

# ── Try multiple encodings (Kaggle files often use latin-1) ──────────────────
for enc in ['utf-8', 'latin-1', 'cp1252', 'utf-8-sig']:
    try:
        df_raw = pd.read_csv(io.BytesIO(uploaded[filename]), encoding=enc, low_memory=False)
        print(f"✅ Loaded with encoding: {enc}")
        break
    except Exception as e:
        continue

print(f"\n📄 File     : {filename}")
print(f"   Shape    : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Columns  : {list(df_raw.columns)}")
print(f"\nFirst 3 rows:")
display(df_raw.head(3))
print(f"\nData types:")
print(df_raw.dtypes)


## Cell 3 — Agent inspects your data + builds a plan

Reads your actual columns, detects types, finds problems, builds action plan.


In [ ]:
class DataAgent:
    def __init__(self, df):
        self.df_raw   = df.copy()
        self.df       = df.copy()
        self.plan     = []
        self.insights = []
        self.numeric_cols = []
        self.cat_cols     = []
        self.date_cols    = []
        self.high_card    = []   # categorical cols with too many unique values to plot
        self.null_cols    = pd.Series(dtype=int)
        self.outlier_cols = {}

    def inspect(self):
        df   = self.df
        rows, cols = df.shape

        print("=" * 60)
        print("  AGENT INSPECTION REPORT")
        print("=" * 60)
        print(f"  Rows    : {rows:,}")
        print(f"  Columns : {cols}")
        print()

        # ── Detect column types ───────────────────────────────────────────────
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

        # Detect date columns from object cols
        for col in df.select_dtypes(include=['object']).columns:
            sample = df[col].dropna().head(30).astype(str)
            try:
                parsed = pd.to_datetime(sample, infer_datetime_format=True, errors='coerce')
                if parsed.notna().mean() > 0.7:   # 70%+ parsed successfully = date col
                    self.date_cols.append(col)
            except:
                pass

        self.cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns
                         if c not in self.date_cols]
        self.high_card = [c for c in self.cat_cols if df[c].nunique() > 25]

        print(f"  Numeric columns ({len(self.numeric_cols)}) :")
        for c in self.numeric_cols:
            print(f"    • {c}  (min={df[c].min():.2f}, max={df[c].max():.2f}, mean={df[c].mean():.2f})")
        print()
        print(f"  Categorical columns ({len(self.cat_cols)}) :")
        for c in self.cat_cols:
            card = df[c].nunique()
            flag = " ⚠️ high cardinality — will skip plot" if c in self.high_card else f"  ({card} unique)"
            print(f"    • {c}{flag}")
        print()
        if self.date_cols:
            print(f"  Date columns ({len(self.date_cols)}) : {self.date_cols}")
            print()

        # ── Missing values ────────────────────────────────────────────────────
        nulls = df.isnull().sum()
        self.null_cols = nulls[nulls > 0]
        total_null_pct = df.isnull().sum().sum() / (rows * cols) * 100
        print(f"  Missing values: {self.null_cols.sum():,} total ({total_null_pct:.1f}% of dataset)")
        if len(self.null_cols):
            for col, cnt in self.null_cols.items():
                action = "→ will DROP column (>50% missing)" if cnt/rows > 0.5 else "→ will fill"
                print(f"    • {col:30s} {cnt:5d} missing ({cnt/rows*100:.1f}%)  {action}")
        else:
            print("    None ✅")
        print()

        # ── Duplicates ────────────────────────────────────────────────────────
        self.dup_count = df.duplicated().sum()
        print(f"  Duplicate rows : {self.dup_count:,}")
        print()

        # ── Outliers (IQR) ────────────────────────────────────────────────────
        for col in self.numeric_cols:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            if IQR == 0:
                continue
            mask = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)
            cnt  = mask.sum()
            if cnt > 0:
                self.outlier_cols[col] = cnt
        print(f"  Outlier columns: {len(self.outlier_cols)}")
        for col, cnt in self.outlier_cols.items():
            print(f"    • {col:30s} {cnt:5d} outliers")
        print()

        # ── Build action plan ─────────────────────────────────────────────────
        print("=" * 60)
        print("  ACTION PLAN")
        print("=" * 60)
        step = 1
        drop_cols = [c for c, n in self.null_cols.items() if n/rows > 0.5]
        if drop_cols:
            self.plan.append(('drop_cols', drop_cols))
            print(f"  Step {step}: Drop columns with >50% missing: {drop_cols}")
            step += 1
        if self.dup_count > 0:
            self.plan.append(('remove_duplicates', None))
            print(f"  Step {step}: Remove {self.dup_count:,} duplicate rows")
            step += 1
        fill_cols = {c:n for c,n in self.null_cols.items() if c not in drop_cols}
        if fill_cols:
            self.plan.append(('handle_nulls', fill_cols))
            print(f"  Step {step}: Fill missing values ({len(fill_cols)} columns)")
            step += 1
        if self.outlier_cols:
            self.plan.append(('cap_outliers', None))
            print(f"  Step {step}: Cap outliers in {len(self.outlier_cols)} columns")
            step += 1
        if self.date_cols:
            self.plan.append(('parse_dates', None))
            print(f"  Step {step}: Parse {self.date_cols} → extract year, month, weekday")
            step += 1
        self.plan.append(('analyse', None))
        print(f"  Step {step}: Run full analysis + generate plots")
        step += 1
        self.plan.append(('export', None))
        print(f"  Step {step}: Export cleaned CSV + report")
        print()
        print(f"  Total steps planned: {len(self.plan)}")


agent = DataAgent(df_raw)
agent.inspect()


## Cell 4 — Agent cleans your data
Executes only the steps it planned in Cell 3.

In [ ]:
def clean(agent):
    df   = agent.df.copy()
    rows_before = len(df)
    print("🔧 Cleaning...\n")

    for step_name, step_data in agent.plan:

        if step_name == 'drop_cols':
            df = df.drop(columns=step_data, errors='ignore')
            # Update numeric/cat lists
            agent.numeric_cols = [c for c in agent.numeric_cols if c not in step_data]
            agent.cat_cols     = [c for c in agent.cat_cols     if c not in step_data]
            print(f"  ✅ Dropped columns with >50% missing : {step_data}")

        elif step_name == 'remove_duplicates':
            before = len(df)
            df = df.drop_duplicates()
            print(f"  ✅ Removed duplicates : {before - len(df):,} rows removed")

        elif step_name == 'handle_nulls':
            for col in step_data:
                if col not in df.columns:
                    continue
                n = df[col].isnull().sum()
                if n == 0:
                    continue
                if col in agent.numeric_cols:
                    val = df[col].median()
                    df[col] = df[col].fillna(val)
                    print(f"  ✅ Filled null (median) : {col:30s}  {n} values → {val:.2f}")
                else:
                    val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
                    df[col] = df[col].fillna(val)
                    print(f"  ✅ Filled null (mode)   : {col:30s}  {n} values → '{val}'")

        elif step_name == 'cap_outliers':
            for col in agent.outlier_cols:
                if col not in df.columns:
                    continue
                lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
                cnt_before = ((df[col] < lo) | (df[col] > hi)).sum()
                df[col] = df[col].clip(lo, hi)
                print(f"  ✅ Capped outliers       : {col:30s}  {cnt_before} values clipped to [{lo:.2f}, {hi:.2f}]")

        elif step_name == 'parse_dates':
            for col in agent.date_cols:
                if col not in df.columns:
                    continue
                try:
                    df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
                    df[col+'_year']    = df[col].dt.year
                    df[col+'_month']   = df[col].dt.month
                    df[col+'_weekday'] = df[col].dt.day_name()
                    print(f"  ✅ Parsed date           : {col:30s}  → added _year, _month, _weekday")
                except Exception as e:
                    print(f"  ⚠️  Could not parse {col}: {e}")

    agent.df = df
    print(f"\n✅ Cleaning complete!")
    print(f"   Rows before : {rows_before:,}")
    print(f"   Rows after  : {len(df):,}")
    print(f"   Nulls left  : {df.isnull().sum().sum()}")
    print(f"\nPreview:")
    display(agent.df.head(3))

clean(agent)


## Cell 5 — Agent analyses your data + generates all plots
Plots are chosen based on YOUR actual columns — not hardcoded.

In [ ]:
def analyse(agent):
    df      = agent.df
    num     = [c for c in agent.numeric_cols if c in df.columns]
    cat     = [c for c in agent.cat_cols if c in df.columns and c not in agent.high_card]
    date    = [c for c in agent.date_cols if c in df.columns]
    insights = []

    print("📊 Generating analysis...\n")

    # ── 1. Statistical summary ────────────────────────────────────────────────
    print("── 1. Statistical summary ──")
    display(df[num].describe().T.round(2))

    # ── 2. Missing values check after cleaning ────────────────────────────────
    remaining_nulls = df.isnull().sum()
    remaining_nulls = remaining_nulls[remaining_nulls > 0]
    if len(remaining_nulls):
        print("\n── 2. Remaining missing values ──")
        display(remaining_nulls.to_frame("count"))

    # ── 3. Distributions for ALL numeric columns ──────────────────────────────
    if num:
        print(f"\n── 3. Distributions ({len(num)} numeric columns) ──")
        ncols  = min(3, len(num))
        nrows  = (len(num) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows))
        axes = np.array(axes).flatten() if hasattr(axes, '__iter__') else [axes]
        for i, col in enumerate(num):
            data = df[col].dropna()
            axes[i].hist(data, bins=min(40, len(data)//5+5),
                         color=PALETTE[i % len(PALETTE)], edgecolor='white', alpha=0.85)
            axes[i].set_title(col)
            axes[i].set_xlabel(col)
            axes[i].set_ylabel("Count")
            skew = data.skew()
            axes[i].text(0.97, 0.95, f"skew={skew:.2f}\nmedian={data.median():.1f}",
                         transform=axes[i].transAxes, ha='right', va='top', fontsize=8,
                         color='#555', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            insights.append(f"'{col}': min={data.min():.2f}, max={data.max():.2f}, mean={data.mean():.2f}, skew={skew:.2f}")
        for j in range(len(num), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle("Numeric column distributions", fontsize=13, fontweight='500', y=1.01)
        plt.tight_layout()
        plt.savefig("plot_distributions.png", dpi=120, bbox_inches='tight')
        plt.show(); plt.close()

    # ── 4. Correlation heatmap ────────────────────────────────────────────────
    if len(num) >= 2:
        print(f"\n── 4. Correlation matrix ({len(num)} numeric columns) ──")
        corr = df[num].corr()
        sz   = max(6, len(num))
        fig, ax = plt.subplots(figsize=(sz, sz-1))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='RdYlGn',
                    center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
                    annot_kws={"size": max(7, 10-len(num))})
        ax.set_title("Correlation matrix")
        plt.tight_layout()
        plt.savefig("plot_correlation.png", dpi=120, bbox_inches='tight')
        plt.show(); plt.close()

        # Top correlations
        c2 = corr.abs().unstack()
        c2 = c2[c2 < 1.0].sort_values(ascending=False).drop_duplicates()
        if len(c2):
            tp = c2.index[0]; tv = c2.iloc[0]
            insights.append(f"Strongest correlation: '{tp[0]}' ↔ '{tp[1]}' (r={tv:.2f})")
            # Print top 5 correlations
            print("  Top correlations:")
            for idx, val in c2.head(5).items():
                print(f"    {idx[0]:25s} ↔ {idx[1]:25s}  r = {val:.3f}")

    # ── 5. ALL categorical columns ────────────────────────────────────────────
    if cat:
        print(f"\n── 5. Categorical value counts ({len(cat)} columns) ──")
        for i, col in enumerate(cat):
            vc  = df[col].value_counts()
            top = min(15, len(vc))
            fig, ax = plt.subplots(figsize=(8, max(3, top*0.35)))
            bars = ax.barh(vc.index[:top].astype(str)[::-1],
                           vc.values[:top][::-1],
                           color=PALETTE[i % len(PALETTE)], alpha=0.85)
            # Add value labels on bars
            for bar, val in zip(bars, vc.values[:top][::-1]):
                ax.text(bar.get_width() + 0.01*vc.values[0], bar.get_y() + bar.get_height()/2,
                        f'{val:,}', va='center', fontsize=9)
            ax.set_title(f"'{col}' — top {top} values  ({df[col].nunique()} unique total)")
            ax.set_xlabel("Count")
            plt.tight_layout()
            plt.savefig(f"plot_cat_{col[:20]}.png", dpi=120, bbox_inches='tight')
            plt.show(); plt.close()
            insights.append(f"'{col}' top value: '{vc.index[0]}' ({vc.values[0]:,} rows, {vc.values[0]/len(df)*100:.1f}%)")

    # ── 6. Boxplots for all numeric columns ───────────────────────────────────
    if num:
        print(f"\n── 6. Boxplots — outlier check ({len(num)} columns) ──")
        ncols = min(4, len(num))
        nrows = (len(num) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows))
        axes = np.array(axes).flatten() if hasattr(axes, '__iter__') else [axes]
        for i, col in enumerate(num):
            axes[i].boxplot(df[col].dropna(), patch_artist=True,
                            boxprops=dict(facecolor=PALETTE[i % len(PALETTE)], alpha=0.6),
                            medianprops=dict(color='black', linewidth=2),
                            flierprops=dict(marker='o', markersize=3, alpha=0.4))
            axes[i].set_title(col)
        for j in range(len(num), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle("Boxplots after cleaning", fontsize=13, fontweight='500', y=1.01)
        plt.tight_layout()
        plt.savefig("plot_boxplots.png", dpi=120, bbox_inches='tight')
        plt.show(); plt.close()

    # ── 7. Categorical × numeric (top 3 combos) ──────────────────────────────
    if cat and num:
        print(f"\n── 7. Categorical × Numeric group analysis ──")
        combos = [(c, n) for c in cat[:3] for n in num[:3]][:6]
        for cat_col, num_col in combos:
            if df[cat_col].nunique() > 20:
                continue
            grp = df.groupby(cat_col)[num_col].agg(['mean','median','count']).round(2)
            grp = grp.sort_values('mean', ascending=False)
            fig, ax = plt.subplots(figsize=(max(6, len(grp)*0.6+2), 4))
            bars = ax.bar(grp.index.astype(str), grp['mean'],
                          color=PALETTE[:len(grp)], alpha=0.85, edgecolor='white')
            for bar, val in zip(bars, grp['mean']):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                        f'{val:.1f}', ha='center', va='bottom', fontsize=9)
            ax.set_title(f"Mean '{num_col}' by '{cat_col}'")
            ax.set_xlabel(cat_col)
            ax.set_ylabel(f"Mean {num_col}")
            plt.xticks(rotation=30, ha='right')
            plt.tight_layout()
            plt.savefig(f"plot_{cat_col[:15]}_{num_col[:15]}.png", dpi=120, bbox_inches='tight')
            plt.show(); plt.close()
            top_grp = grp.index[0]
            insights.append(f"Highest mean '{num_col}' in '{cat_col}': '{top_grp}' = {grp['mean'].iloc[0]:.2f}")

    # ── 8. Time trends (if date columns found) ────────────────────────────────
    month_cols = [c for c in df.columns if c.endswith('_month')]
    if month_cols and num:
        print(f"\n── 8. Time trends (monthly) ──")
        m_col = month_cols[0]
        for num_col in num[:3]:
            try:
                monthly = df.groupby(m_col)[num_col].mean()
                fig, ax = plt.subplots(figsize=(9, 4))
                ax.plot(monthly.index, monthly.values, marker='o',
                        color=PALETTE[0], linewidth=2, markersize=6)
                ax.fill_between(monthly.index, monthly.values, alpha=0.12, color=PALETTE[0])
                ax.set_xlabel("Month")
                ax.set_ylabel(f"Avg {num_col}")
                ax.set_title(f"Monthly average of '{num_col}'")
                ax.set_xticks(monthly.index)
                plt.tight_layout()
                plt.savefig(f"plot_trend_{num_col[:20]}.png", dpi=120, bbox_inches='tight')
                plt.show(); plt.close()
                peak = monthly.idxmax()
                insights.append(f"Peak month for '{num_col}': Month {peak} (avg={monthly[peak]:.2f})")
            except Exception as e:
                print(f"  Could not plot trend for {num_col}: {e}")

    # ── 9. Pairplot (top 4 numeric cols if <= 5 numeric cols) ────────────────
    if 2 <= len(num) <= 6:
        print(f"\n── 9. Pairplot (relationships between all numeric columns) ──")
        plot_num = num[:4]
        pair_df = df[plot_num].dropna()
        if len(pair_df) > 1000:
            pair_df = pair_df.sample(1000, random_state=42)
        hue_col = cat[0] if cat and df[cat[0]].nunique() <= 8 else None
        if hue_col:
            pair_df[hue_col] = df.loc[pair_df.index, hue_col]
        try:
            g = sns.pairplot(pair_df, hue=hue_col, corner=True,
                             plot_kws={'alpha':0.5, 's':15},
                             diag_kws={'bins':20})
            g.fig.suptitle("Pairplot — numeric relationships", y=1.01, fontsize=13)
            plt.savefig("plot_pairplot.png", dpi=100, bbox_inches='tight')
            plt.show(); plt.close()
        except Exception as e:
            print(f"  Pairplot skipped: {e}")

    # ── Summary insights ──────────────────────────────────────────────────────
    agent.insights = insights
    print("\n" + "="*60)
    print("  KEY INSIGHTS")
    print("="*60)
    for i, ins in enumerate(insights, 1):
        print(f"  {i:2d}. {ins}")

analyse(agent)


## Cell 6 — Export cleaned CSV + report

In [ ]:
def export(agent):
    df = agent.df
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Cleaned CSV
    csv_name = f"cleaned_{ts}.csv"
    df.to_csv(csv_name, index=False)
    files.download(csv_name)
    print(f"✅ Cleaned CSV  : {csv_name}  ({len(df):,} rows × {df.shape[1]} cols)")

    # Report
    rep_name = f"report_{ts}.txt"
    lines = [
        "DATA ANALYSIS AGENT — REPORT",
        f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "=" * 60,
        "",
        "DATASET",
        f"  Raw rows     : {len(agent.df_raw):,}",
        f"  Cleaned rows : {len(df):,}",
        f"  Columns      : {list(df.columns)}",
        "",
        "CLEANING STEPS TAKEN",
    ]
    for step_name, _ in agent.plan:
        if step_name not in ('analyse','export'):
            lines.append(f"  - {step_name.replace('_',' ').title()}")
    lines += ["", "NUMERIC STATISTICS", ""]
    num = [c for c in agent.numeric_cols if c in df.columns]
    if num:
        lines.append(df[num].describe().round(3).to_string())
    lines += ["", "KEY INSIGHTS", ""]
    for i, ins in enumerate(agent.insights, 1):
        lines.append(f"  {i}. {ins}")

    with open(rep_name, 'w') as f:
        f.write("\n".join(lines))
    files.download(rep_name)
    print(f"✅ Report       : {rep_name}")

export(agent)


## Cell 7 — Custom queries on your data

`agent.df` is your cleaned DataFrame.
Write any Pandas operation here.


In [ ]:
df = agent.df   # your cleaned data

print("Your columns:")
print(list(df.columns))
print()
print("Shape:", df.shape)
print()
display(df.head(5))


In [ ]:
# ── Write your own analysis here ─────────────────────────────────────────────
# Examples:
#
# df['your_column'].value_counts()
# df.groupby('category_col')['numeric_col'].mean().sort_values(ascending=False)
# df[df['numeric_col'] > 100].shape[0]
# df.corr()['target_column'].sort_values()

# Replace this line with your own query:
print("Write your custom query above — agent.df is your cleaned DataFrame.")


## Cell 8 — Resume bullet

> **Autonomous Data Analysis Agent** | Python · Pandas · Seaborn · Matplotlib · SciPy  
> Built an agentic data pipeline that autonomously inspects any uploaded CSV,
> decides which cleaning steps to execute (null imputation, deduplication, outlier
> capping, date parsing), generates 9 types of adaptive visualisations (distributions,
> correlations, pairplot, categorical breakdowns, time trends, group comparisons),
> and exports a cleaned dataset + summary report — fully adapts to any dataset structure.

---
*Built by Sahil Prakash — [github.com/Sahilp008](https://github.com/Sahilp008)*
